# ESTIA

In [ ]:
import scipp as sc
import tof

import scippnexus as snx
from ess.reduce.unwrap import GenericUnwrapWorkflow
from ess.reduce.nexus.types import *
from ess.reduce.unwrap.types import *
from ess.reduce.unwrap.lut import LtotalRange, ChopperFrameSequence

## Chopper parameters

In [ ]:
min_wavelength = sc.scalar(3.75, unit="angstrom")
chopper_position = sc.scalar(10.895, unit="m")
max_velocity = (sc.constants.h / sc.constants.m_n / min_wavelength).to(unit="m/s")
delay = chopper_position / max_velocity
pulse_stride = 1
frequency = -sc.scalar(14.0, unit="Hz") / pulse_stride

estia_choppers = {
    "fc": DiskChopper(
        frequency=frequency,
        beam_position=sc.scalar(0.0, unit="deg"),
        phase=delay * frequency * sc.scalar(360, unit="deg"),
        axle_position=sc.vector(
            value=[0, 0.0, chopper_position.value], unit=chopper_position.unit
        ),
        slit_begin=sc.array(
            dims=["cutout"],
            values=[0.0],
            unit="deg",
        ),
        slit_end=sc.array(
            dims=["cutout"],
            values=[98.0],
            unit="deg",
        ),
    ),
}

In [ ]:
estia_choppers["fc"]

## Tof model

In [ ]:
source = tof.Source(facility="ess", neutrons=1_000_000, pulses=2)
source_position = sc.vector([0, 0, 0], unit="m")
detector = tof.Detector(distance=sc.scalar(45.0, unit="m"), name="detector")

params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in estia_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

## Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = estia_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = sc.scalar(5, unit="m"), detector.distance
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot() + table.array["distance", -1].plot(errorbars="band")

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()